# Postprocess GSA of hydrogeophysical model (v5.6, With MRS undetectable water V2 and corrected geometry)

In [1]:
import numpy as np
from uqpylab import sessions

#STEP 1 : start UQ session
# Uncomment and fill with your UQ[py]Lab credentials
myToken = 'd0e7d6557734ff749ce60fb8ec9fdcd956c09f4b' # The user's token to access the UQCloud API
UQCloud_instance = 'https://uqcloud.ethz.ch' # The UQCloud instance to use

# Start the session
mySession = sessions.cloud(host=UQCloud_instance, token=myToken, force_restart=True)
mySession.timeout = 5000
# (Optional) Get a convenient handle to the command line interface
uq = mySession.cli
# Reset the session
mySession.reset()

Processing . done!

 uqpylab.sessions :: INFO     :: Reset successful.
 uqpylab.sessions :: INFO     :: Succesfully restarted the worker.
 uqpylab.sessions :: INFO     :: This is UQ[py]Lab, version 1.0, running on https://uqcloud.ethz.ch. 
                                 UQ[py]Lab is free software, published under the open source BSD 3-clause license.
                                 To request special permissions, please contact:
                                  - Stefano Marelli (marelli@ibk.baug.ethz.ch).
                                 A new session (7766e17cd4ed4146902d342ca055ba32) started.
 uqpylab.sessions :: INFO     :: Reset successful.


In [3]:
#STEP2 : Retrieve simulation data
simfiles_dir = '../Hydro_MRS_GSA_numexp/Res_sim/'
fn_st = 'GSA5_5_'
n = '1'

# first set of simulation results
Y_E1 = np.loadtxt(simfiles_dir+fn_st+n+'_E1.csv', delimiter = ',')
Y_E2 = np.loadtxt(simfiles_dir+fn_st+n+'_E2.csv', delimiter = ',')
Y_E3 = np.loadtxt(simfiles_dir+fn_st+n+'_E3.csv', delimiter = ',')
Y_E4 = np.loadtxt(simfiles_dir+fn_st+n+'_E4.csv', delimiter = ',')
Y_E5 = np.loadtxt(simfiles_dir+fn_st+n+'_E5.csv', delimiter = ',')

X = np.loadtxt(simfiles_dir+fn_st+n+'_X.csv', delimiter = ',')

In [4]:
Nsim_or = Y_E1.shape[0]

In [5]:
nanidx = np.argwhere(np.isnan(Y_E1[:,0]))

Y_E1 = np.delete(Y_E1, nanidx, 0)
Y_E2 = np.delete(Y_E2, nanidx, 0)
Y_E3 = np.delete(Y_E3, nanidx, 0)
Y_E4 = np.delete(Y_E4, nanidx, 0)
Y_E5 = np.delete(Y_E5, nanidx, 0)
X = np.delete(X, nanidx, 0)

In [6]:
Nsim_success = Y_E1.shape[0] # Number of simulations after deleting NaN (non-convergent simulations)

In [7]:
print(str(nanidx.shape[0])+'/'+str(Nsim_or)+' simulations did not converge ('+str((nanidx.shape[0]/Nsim_or)*100)[:4]+' %).')

39/3000 simulations did not converge (1.3 %).


In [8]:
#split into calibration/validation set (here, size : 0.75 / 0.25)

Ncal = int(3*(int(X.shape[0]/4)))

X_cal = X[:Ncal]
Y_E1_cal = Y_E1[:Ncal]
Y_E2_cal = Y_E2[:Ncal]
Y_E3_cal = Y_E3[:Ncal]
Y_E4_cal = Y_E4[:Ncal]
Y_E5_cal = Y_E5[:Ncal]



X_val = X[Ncal:]
Y_E1_val = Y_E1[Ncal:]
Y_E2_val = Y_E2[Ncal:]
Y_E3_val = Y_E3[Ncal:]
Y_E4_val = Y_E4[Ncal:]
Y_E5_val = Y_E5[Ncal:]

In [9]:
print(X_cal.shape)
print(Y_E1_cal.shape)

(2220, 18)
(2220, 66)


In [10]:
print(X_val.shape)
print(Y_E1_val.shape)

(741, 18)
(741, 66)


In [11]:
# STEP 3 : probabilistic model input 

InputOpts = {
    "Marginals": [
        # Soil Layer 
        {"Type": "Uniform",
         "Parameters": [0.00, 0.10] # tr1
        },
        {"Type": "Uniform",
         "Parameters": [0.30, 0.50] #ts1
        },
        {"Type": "Uniform",
         "Parameters": [1e-2, 0.15] # alpha1
        },
        {"Type": "Uniform",
         "Parameters": [1.1, 3] # n1
        },
        {"Type": "Uniform",
         "Parameters": [-1., 2.18] # log10Ks1
        },
        # Saprolite Layer
        {"Type": "Uniform",
         "Parameters": [0.00, 0.04] # tr2
        },
        {"Type": "Uniform",
         "Parameters": [0.08, 0.20] #ts2
        },
        {"Type": "Uniform",
         "Parameters": [1e-2, 0.15] # alpha2
        },
        {"Type": "Uniform",
         "Parameters": [1.1, 3] # n2
        },
        {"Type": "Uniform",
         "Parameters": [-1, 0.18] # log10Ks2
        },
        # MRS undetectable water parameters
        # Soil layer
        {"Type": "Uniform",
         "Parameters": [0.20, 0.45] # theta_i_percent_base
        },
        {"Type": "Uniform",
         "Parameters": [0.40, 0.50] # S_lim
        },
        {"Type": "Uniform",
         "Parameters": [0.50, 0.70] # theta_i_percent_max
        },
        # Saprolite layer
        {"Type": "Uniform",
         "Parameters": [0.20, 0.45] # theta_i_percent_base
        },
        {"Type": "Uniform",
         "Parameters": [0.40, 0.50] # S_lim
        },
        {"Type": "Uniform",
         "Parameters": [0.50, 0.70] # theta_i_percent_max
        },

        # Non simulated zone parameters
        {"Type": "Uniform",
         "Parameters": [0.01, 0.05] # tMRS_Bedrock
        },
        # delta_h piezo
        {"Type": "Uniform",
         "Parameters": [-20.0, 20.0] 
        },
    ]
}

myInput = uq.createInput(InputOpts)

## Output 1 : $E_0(q=60.0A.ms)$

In [12]:
#STEP4 : PCE metamodel
# Select PCE as the metamodeling tool:

MetaOpts = {
    'Type': 'Metamodel',
    'MetaType': 'PCE'
}

# Use experimental design loaded from the data files:


MetaOpts['ExpDesign'] = {
    'X': X_cal.tolist(),
    'Y': Y_E1_cal.tolist()
}

#Set the maximum polynomial degree to 5:

MetaOpts["Degree"] = np.arange(1,5).tolist()

# Specify the parameters of the PCE truncation scheme

#MetaOpts['TruncOptions'] = {
#    'qNorm': 0.7,
#    'MaxInteraction': 2
#}


#Provide the validation data set to get the validation error:

MetaOpts['ValidationSet'] = {
    'X': X_val.tolist(),
    'Y': Y_E1_val.tolist()
}

# Create the PCE metamodel:

myPCE = uq.createModel(MetaOpts)

Processing .................................................................................................................................................................... done!



In [13]:
# Print a summary of the resulting PCE metamodel:
uq.print(myPCE)

printed 
> In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_PCE_print', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m', 12)" style="font-weight:bold">uq_PCE_print</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m',12,0)">line 12</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_print_uq_metamodel', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m', 16)" style="font-weight:bold">uq_print_uq_metamodel</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m',16,0)">line 16</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_module/Print', '/root/.mcrCache9.14/uq

In [14]:
LOO_errors = []
for i in range(len(myPCE['Error'])):
    LOO_errors.append(myPCE['Error'][i]['LOO'])

max(LOO_errors)

0.007223274440513272

In [15]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol'
}

SobolOpts['Sobol'] = {
    'Order': 2
}

SobolOpts['Sobol']['SampleSize'] = 1e5; #1e5

mySobolAnalysisPCE = uq.createAnalysis(SobolOpts)

mySobolResultsPCE = mySobolAnalysisPCE['Results']

In [16]:
import pickle
#create a binary pickle file 
dir = "Data/"
f = open(dir+"PCE_E1.pkl", "wb")
pickle.dump(myPCE, f)
f.close()

f = open(dir+"Sobol_E1.pkl", "wb")
pickle.dump(mySobolResultsPCE, f)
f.close()

## Output 2 : $E_0(q=144.61A.ms)$

In [17]:
MetaOpts = {
    'Type': 'Metamodel',
    'MetaType': 'PCE'
}

# Use experimental design loaded from the data files:


MetaOpts['ExpDesign'] = {
    'X': X_cal.tolist(),
    'Y': Y_E2_cal.tolist()
}

#Set the maximum polynomial degree to 5:

MetaOpts["Degree"] = np.arange(1,5).tolist()

# Specify the parameters of the PCE truncation scheme

#MetaOpts['TruncOptions'] = {
#    'qNorm': 0.7,
#    'MaxInteraction': 2
#}



#Provide the validation data set to get the validation error:

MetaOpts['ValidationSet'] = {
    'X': X_val.tolist(),
    'Y': Y_E2_val.tolist()
}

# Create the PCE metamodel:

myPCE_E2 = uq.createModel(MetaOpts)

Processing ................................................................................................................................................................. done!



In [18]:
# Print a summary of the resulting PCE metamodel:
uq.print(myPCE_E2)

printed 
> In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_PCE_print', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m', 12)" style="font-weight:bold">uq_PCE_print</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m',12,0)">line 12</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_print_uq_metamodel', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m', 16)" style="font-weight:bold">uq_print_uq_metamodel</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m',16,0)">line 16</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_module/Print', '/root/.mcrCache9.14/uq

In [19]:
LOO_errors_2 = []
for i in range(len(myPCE_E2['Error'])):
    LOO_errors_2.append(myPCE_E2['Error'][i]['LOO'])

max(LOO_errors_2)

0.011884918143983743

In [20]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol'
}

SobolOpts['Sobol'] = {
    'Order': 2
}

SobolOpts['Sobol']['SampleSize'] = 1e5; #1e5

mySobolAnalysisPCE_E2 = uq.createAnalysis(SobolOpts)

mySobolResultsPCE_E2 = mySobolAnalysisPCE_E2['Results']

In [21]:
#create a binary pickle file 
dir = "Data/"
f = open(dir+"PCE_E2.pkl", "wb")
pickle.dump(myPCE_E2, f)
f.close()

f = open(dir+"Sobol_E2.pkl", "wb")
pickle.dump(mySobolResultsPCE_E2, f)
f.close()

## Output 3 : $E_0(q=361.55A.ms)$

In [22]:
MetaOpts = {
    'Type': 'Metamodel',
    'MetaType': 'PCE'
}

# Use experimental design loaded from the data files:


MetaOpts['ExpDesign'] = {
    'X': X_cal.tolist(),
    'Y': Y_E3_cal.tolist()
}

#Set the maximum polynomial degree to 5:

MetaOpts["Degree"] = np.arange(1,5).tolist()

# Specify the parameters of the PCE truncation scheme

#MetaOpts['TruncOptions'] = {
#    'qNorm': 0.7,
#    'MaxInteraction': 2
#}



#Provide the validation data set to get the validation error:

MetaOpts['ValidationSet'] = {
    'X': X_val.tolist(),
    'Y': Y_E3_val.tolist()
}

# Create the PCE metamodel:

myPCE_E3 = uq.createModel(MetaOpts)
# Print a summary of the resulting PCE metamodel:
uq.print(myPCE_E3)

Processing ................................................................................................................................................................... done!

printed 
> In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_PCE_print', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m', 12)" style="font-weight:bold">uq_PCE_print</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m',12,0)">line 12</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_print_uq_metamodel', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m', 16)" style="font-weight:bold">uq_print_uq_metamodel</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/b

In [23]:
LOO_errors_3 = []
for i in range(len(myPCE_E3['Error'])):
    LOO_errors_3.append(myPCE_E3['Error'][i]['LOO'])

max(LOO_errors_3)

0.0027681360418427207

In [24]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol'
}

SobolOpts['Sobol'] = {
    'Order': 2
}

SobolOpts['Sobol']['SampleSize'] = 1e5; #1e5

mySobolAnalysisPCE_E3 = uq.createAnalysis(SobolOpts)

mySobolResultsPCE_E3 = mySobolAnalysisPCE_E3['Results']

#create a binary pickle file 
dir = "Data/"
f = open(dir+"PCE_E3.pkl", "wb")
pickle.dump(myPCE_E3, f)
f.close()

f = open(dir+"Sobol_E3.pkl", "wb")
pickle.dump(mySobolResultsPCE_E3, f)
f.close()

## Output 4 : $E_0(q=903.94 A.ms)$

In [25]:
MetaOpts = {
    'Type': 'Metamodel',
    'MetaType': 'PCE'
}

# Use experimental design loaded from the data files:


MetaOpts['ExpDesign'] = {
    'X': X_cal.tolist(),
    'Y': Y_E4_cal.tolist()
}

#Set the maximum polynomial degree to 5:

MetaOpts["Degree"] = np.arange(1,5).tolist()

# Specify the parameters of the PCE truncation scheme

#MetaOpts['TruncOptions'] = {
#    'qNorm': 0.7,
#    'MaxInteraction': 2
#}



#Provide the validation data set to get the validation error:

MetaOpts['ValidationSet'] = {
    'X': X_val.tolist(),
    'Y': Y_E4_val.tolist()
}

# Create the PCE metamodel:

myPCE_E4 = uq.createModel(MetaOpts)
# Print a summary of the resulting PCE metamodel:
uq.print(myPCE_E4)


Processing ................................................................................................................................................................ done!

printed 
> In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_PCE_print', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m', 12)" style="font-weight:bold">uq_PCE_print</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m',12,0)">line 12</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_print_uq_metamodel', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m', 16)" style="font-weight:bold">uq_print_uq_metamodel</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/buil

In [26]:
LOO_errors_4 = []
for i in range(len(myPCE_E4['Error'])):
    LOO_errors_4.append(myPCE_E4['Error'][i]['LOO'])

max(LOO_errors_4)

0.0005624568219204784

In [27]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol'
}

SobolOpts['Sobol'] = {
    'Order': 2
}

SobolOpts['Sobol']['SampleSize'] = 1e5; #1e5

mySobolAnalysisPCE_E4 = uq.createAnalysis(SobolOpts)

mySobolResultsPCE_E4 = mySobolAnalysisPCE_E4['Results']

#create a binary pickle file 
dir = "Data/"
f = open(dir+"PCE_E4.pkl", "wb")
pickle.dump(myPCE_E4, f)
f.close()

f = open(dir+"Sobol_E4.pkl", "wb")
pickle.dump(mySobolResultsPCE_E4, f)
f.close()

## Output 5 : $E_0(q=2260.00 A.ms)$

In [28]:
MetaOpts = {
    'Type': 'Metamodel',
    'MetaType': 'PCE'
}

# Use experimental design loaded from the data files:


MetaOpts['ExpDesign'] = {
    'X': X_cal.tolist(),
    'Y': Y_E5_cal.tolist()
}

#Set the maximum polynomial degree to 5:

MetaOpts["Degree"] = np.arange(1,5).tolist()

# Specify the parameters of the PCE truncation scheme

#MetaOpts['TruncOptions'] = {
#    'qNorm': 0.7,
#    'MaxInteraction': 2
#}


#Provide the validation data set to get the validation error:

MetaOpts['ValidationSet'] = {
    'X': X_val.tolist(),
    'Y': Y_E5_val.tolist()
}

# Create the PCE metamodel:

myPCE_E5 = uq.createModel(MetaOpts)
# Print a summary of the resulting PCE metamodel:
uq.print(myPCE_E5)

Processing ...................................................................................................................................................................... done!

printed 
> In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_PCE_print', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m', 12)" style="font-weight:bold">uq_PCE_print</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/PCE/uq_PCE_print.m',12,0)">line 12</a>)
In <a href="matlab:matlab.internal.language.introspective.errorDocCallback('uq_print_uq_metamodel', '/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_model/builtin/uq_metamodel/uq_print_uq_metamodel.m', 16)" style="font-weight:bold">uq_print_uq_metamodel</a> (<a href="matlab: opentoline('/root/.mcrCache9.14/uq_web0/uq_web_cli_f/uqlab_deployable/modules/uq_mode

In [29]:
LOO_errors_5 = []
for i in range(len(myPCE_E5['Error'])):
    LOO_errors_5.append(myPCE_E5['Error'][i]['LOO'])

max(LOO_errors_5)

8.521658894428328e-05

In [30]:
SobolOpts = {
    'Type': 'Sensitivity',
    'Method': 'Sobol'
}

SobolOpts['Sobol'] = {
    'Order': 2
}

SobolOpts['Sobol']['SampleSize'] = 1e5; #1e5

mySobolAnalysisPCE_E5 = uq.createAnalysis(SobolOpts)

mySobolResultsPCE_E5 = mySobolAnalysisPCE_E5['Results']

#create a binary pickle file 
dir = "Data/"
f = open(dir+"PCE_E5.pkl", "wb")
pickle.dump(myPCE_E5, f)
f.close()

f = open(dir+"Sobol_E5.pkl", "wb")
pickle.dump(mySobolResultsPCE_E5, f)
f.close()